# Projekt · Woche 11 · Kontext pro Gespräch im Griff
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Ziel dieser Woche: Ihr messt den **Kontext eines typischen Gesprächs** mit eurem Agenten, baut **Kompaktierung**, **gekürzte Tool-Ergebnisse** und ein **Langzeit-Memory mit Datenschutzregel** ein und vergleicht vorher und nachher. Zur Einordnung rechnen wir aus, was das Gespräch bei einem kommerziellen Anbieter kosten würde.

1. Tools wie in Woche 10 (eigenes Tool einfügen)
2. Ein typisches Gespräch mit acht Fragen festlegen (Beispiele für euren Track vorhanden)
3. Vorher messen: Kontext pro Frage, Tokens pro Gespräch
4. Nachher messen: mit gekürzten Tool-Ergebnissen und Kompaktierung
5. Memory mit `remember`, `recall` und Datenschutzfilter, über zwei Gespräche testen
6. Ergebnisse speichern

In [ ]:
%pip install -q sentence-transformers==6.1.0 openai==3.24.0

In [ ]:
import os, glob, zipfile, csv, json, re, datetime
import numpy as np

ORDNER = "daten"
if not os.path.isdir(ORDNER):
    from google.colab import files
    hochgeladen = files.upload()
    os.makedirs(ORDNER, exist_ok=True)
    for name in hochgeladen:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(ORDNER)
        else:
            os.replace(name, os.path.join(ORDNER, name))

TRACK = "B" if os.path.exists(f"{ORDNER}/wartungshandbuch.md") else "A"
HEUTE = datetime.date.today()
print("Track", TRACK, "| heute:", HEUTE)

def lies_csv(name):
    with open(f"{ORDNER}/{name}", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [ ]:
from sentence_transformers import SentenceTransformer

def chunk_struktur(markdown, dokument):
    chunks, pfad, zeilen, titel = [], {}, [], dokument
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = dokument + " › " + " › ".join(pfad[e] for e in sorted(pfad))
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = []
for d in sorted(glob.glob(f"{ORDNER}/**/*.md", recursive=True)):
    if not os.path.basename(d).lower().startswith("readme"):
        CHUNKS += chunk_struktur(open(d, encoding="utf-8").read(), os.path.basename(d))
modell = SentenceTransformer("intfloat/multilingual-e5-small")
E = modell.encode(["passage: " + c["titel"] + "\n" + c["text"] for c in CHUNKS], normalize_embeddings=True)
print(len(CHUNKS), "Chunks in der Wissensbasis")

def search_knowledge_base(query, k=4):
    q = modell.encode(["query: " + query], normalize_embeddings=True)[0]
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in np.argsort(E @ q)[::-1][:k]]

THEMEN = "Wartungshandbuch der Nordwerk GmbH (Intervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes)" if TRACK == "B" \
         else "Satzung und Sitzungsprotokolle der Datenfabrik Karlsruhe e.V. (Regeln, Beschlüsse, Budgets, Zuständigkeiten)"
TOOL_SUCHE = {
    "name": "search_knowledge_base",
    "description": f"Durchsucht {THEMEN} und liefert die passendsten Abschnitte mit Quelle. Nutze es für Fragen zu Regeln, Abläufen und dokumentiertem Wissen.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Suchanfrage in natürlicher Sprache."}}, "required": ["query"]},
}

In [ ]:
FUNKTIONEN = {"search_knowledge_base": search_knowledge_base}
TOOLS = [TOOL_SUCHE]
SCHREIBENDE_TOOLS = set()

if TRACK == "B":
    MASCHINEN = {m["id"]: m for m in lies_csv("maschinen.csv")}
    TICKETS = lies_csv("stoerungen.csv")
    TICKET_LOG = []

    def _pruefe(machine_id):
        if machine_id not in MASCHINEN:
            raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig: {', '.join(MASCHINEN)}.")

    def get_machine_status(machine_id):
        _pruefe(machine_id)
        m = MASCHINEN[machine_id]
        offen = sum(t["maschine"] == machine_id and t["status"] == "offen" for t in TICKETS)
        return {**m, "offene_tickets": offen}

    def list_tickets(machine_id=None, status="offen"):
        if machine_id: _pruefe(machine_id)
        return [t for t in TICKETS + TICKET_LOG if (machine_id is None or t["maschine"] == machine_id) and (status == "alle" or t["status"] == status)]

    def create_ticket(machine_id, description, priority):
        _pruefe(machine_id)
        ticket = {"ticket": f"T-{1001 + len(TICKET_LOG)}", "datum": HEUTE.isoformat(), "maschine": machine_id,
                  "prioritaet": priority, "status": "offen", "beschreibung": description}
        TICKET_LOG.append(ticket)
        return {"ticket": ticket["ticket"], "status": "angelegt"}

    ID_TEXT = "Maschinen-ID im Format M-01 bis M-08."
    FUNKTIONEN |= {"get_machine_status": get_machine_status, "list_tickets": list_tickets, "create_ticket": create_ticket}
    TOOLS += [
        {"name": "get_machine_status", "description": "Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, Standort, offene Tickets.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}}, "required": ["machine_id"], "additionalProperties": False}},
        {"name": "list_tickets", "description": "Listet Störungstickets, standardmäßig nur offene. Optional für eine Maschine; status 'alle' zeigt auch geschlossene.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "status": {"type": "string", "enum": ["offen", "geschlossen", "alle"]}}}},
        {"name": "create_ticket", "description": "Legt ein neues Störungsticket an. Nur verwenden, wenn der Nutzer ausdrücklich ein Ticket möchte. Prioritäten: hoch = Maschine steht oder Sicherheitsrisiko, mittel = eingeschränkt, niedrig = kein Einfluss.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "description": {"type": "string"}, "priority": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]}},
                          "required": ["machine_id", "description", "priority"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"create_ticket"}

else:
    TERMINE = lies_csv("termine.csv")
    MITGLIEDER = {m["id"]: m for m in lies_csv("mitglieder.csv")}

    def _semester_index(text):
        jahr = int(re.search(r"\d{4}", text).group())
        return jahr * 2 + (1 if text.startswith("WS") else 0)

    def list_events(ab_datum=None, art=None):
        ab = ab_datum or HEUTE.isoformat()
        return [t for t in TERMINE if t["datum"] >= ab and (art is None or t["art"].lower() == art.lower())][:8]

    def get_member_engagement(member_id):
        if member_id not in MITGLIEDER:
            raise ValueError(f"Unbekannte Mitglieds-ID '{member_id}'. Format: D-001.")
        m = MITGLIEDER[member_id]
        aktuell = _semester_index(f"WS {HEUTE.year}" if HEUTE.month >= 10 else f"SoSe {HEUTE.year}")
        # Bewusst nur die Daten, die für eine Bescheinigung nötig sind: keine E-Mail, kein Studiengang
        return {"id": m["id"], "name": f"{m['vorname']} {m['nachname']}".strip(), "rolle": m["rolle"], "ressort": m["ressort"],
                "eintritt": m["eintritt"], "semester_seit_eintritt": aktuell - _semester_index(m["eintritt"]) + 1,
                "engagement_stunden": int(m["engagement_stunden"]), "status": m["status"]}

    def draft_certificate(member_id, taetigkeiten):
        d = get_member_engagement(member_id)
        os.makedirs("bescheinigungen", exist_ok=True)
        pfad = f"bescheinigungen/{member_id}_entwurf.md"
        with open(pfad, "w", encoding="utf-8") as f:
            f.write(f"# Engagementbescheinigung (Entwurf)\n\n{d['name']} ist seit dem {d['eintritt']} Mitglied der Datenfabrik Karlsruhe e.V. "
                    f"und hat sich in dieser Zeit mit rund {d['engagement_stunden']} Stunden engagiert, zuletzt als {d['rolle']}.\n\n"
                    f"Wesentliche Tätigkeiten: {taetigkeiten}\n\nKarlsruhe, {HEUTE:%d.%m.%Y}\n\n(Unterschrift zweier Vorstandsmitglieder)\n")
        return {"entwurf": pfad, "status": "angelegt"}

    FUNKTIONEN |= {"list_events": list_events, "get_member_engagement": get_member_engagement, "draft_certificate": draft_certificate}
    TOOLS += [
        {"name": "list_events", "description": "Listet kommende Termine des Vereins ab einem Datum (Standard: heute), optional nach Art (Sitzung, Workshop, Event, Exkursion, Vortrag, Mitgliederversammlung).",
         "parameters": {"type": "object", "properties": {"ab_datum": {"type": "string", "description": "Datum im Format JJJJ-MM-TT."}, "art": {"type": "string"}}}},
        {"name": "get_member_engagement", "description": "Liefert die für eine Engagementbescheinigung nötigen Daten eines Mitglieds: Name, Rolle, Eintritt, Semester, Engagementstunden, Status. Keine Kontaktdaten.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Mitglieds-ID im Format D-001."}}, "required": ["member_id"], "additionalProperties": False}},
        {"name": "draft_certificate", "description": "Erstellt einen Entwurf einer Engagementbescheinigung als Datei. Vorher mit get_member_engagement und den Regeln aus der Satzung (§ 13) prüfen, ob die Voraussetzungen erfüllt sind.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string"}, "taetigkeiten": {"type": "string", "description": "Wesentliche Tätigkeiten in einem Satz."}},
                          "required": ["member_id", "taetigkeiten"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"draft_certificate"}

print("Tools:", [t["name"] for t in TOOLS], "| schreibend:", SCHREIBENDE_TOOLS)

In [ ]:
# ▶ Euer eigenes Tool aus Woche 8 (Funktion, FUNKTIONEN-Eintrag, TOOLS-Eintrag, ggf. SCHREIBENDE_TOOLS)

## 2 · Gesprächs-Agent mit Messung

In [ ]:
import math, time, copy
from openai import OpenAI
import matplotlib.pyplot as plt
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL = "kit.mistral-small-4-119b-a8b"
PREISE = {"input": 1.00, "cache_write": 1.25, "cache_read": 0.10, "output": 5.00}   # kommerzielle Referenz, $ pro 1 Mio. Tokens

SYSTEM = (f"Du bist der Agent unseres Projekts (Track {TRACK}). Heute ist der {HEUTE:%d.%m.%Y}. "
          "Nutze die Tools, statt Fakten zu raten. Fehlt eine Angabe, frage nach. Nenne bei Wissen aus Dokumenten die Quelle. "
          "Gib keine Kontaktdaten von Personen heraus. Antworte knapp auf Deutsch.")

def kosten_cent(z):
    return sum(z[a] * PREISE[a] for a in PREISE) / 1e6 * 100

def fuehre_tool_aus(name, eingabe):
    if name in SCHREIBENDE_TOOLS:
        return {"status": "in der Messung nicht ausgeführt, als Vorschlag notiert"}, False
    try:
        if name not in FUNKTIONEN:
            raise ValueError(f"Unbekanntes Tool '{name}'. Verfügbar: {', '.join(FUNKTIONEN)}.")
        return FUNKTIONEN[name](**eingabe), False
    except Exception as e:
        return {"fehler": str(e)}, True

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def argumente(tc):
    """Die Argumente eines Tool-Aufrufs kommen als JSON-String."""
    try:
        return json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"_ungueltig": tc.function.arguments}

def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    inhalt = json.dumps(ergebnis, ensure_ascii=False)
    return {"role": "tool", "tool_call_id": tool_call_id, "content": ("FEHLER: " + inhalt) if is_error else inhalt}

def finde_schnitt(nachrichten, behalte_letzte=2):
    fragen = [i for i, n in enumerate(nachrichten) if n["role"] == "user"]
    return fragen[-behalte_letzte] if len(fragen) > behalte_letzte else 0

def als_text(nachrichten):
    zeilen = []
    for n in nachrichten:
        if n["role"] == "tool":
            zeilen.append(f"Tool-Ergebnis: {n['content'][:600]}"); continue
        if n.get("content"):
            zeilen.append(f"{n['role']}: {n['content']}")
        for tc in n.get("tool_calls", []):
            zeilen.append(f"Tool-Aufruf: {tc['function']['name']}({tc['function']['arguments']})")
    return "\n".join(zeilen)

SYSTEM_ZUSAMMENFASSUNG = ("Fasse das Gespräch für die Fortsetzung zusammen. Behalte Befunde, Daten, Zahlen, Entscheidungen, vorgeschlagene "
                          "Maßnahmen, offene Punkte und Fragen des Nutzers. Lass Begrüßungen und vollständige Tool-Ausgaben weg. "
                          "Stichpunkte, höchstens 150 Wörter.")

def kompaktiere(nachrichten, behalte_letzte=2):
    schnitt = finde_schnitt(nachrichten, behalte_letzte)
    if schnitt == 0:
        return nachrichten
    z = client.chat.completions.create(model=MODEL, max_tokens=400,
                                       messages=[{"role": "system", "content": SYSTEM_ZUSAMMENFASSUNG},
                                                 {"role": "user", "content": als_text(nachrichten[:schnitt])}]).choices[0].message.content
    return [{"role": "user", "content": "Zusammenfassung des bisherigen Gesprächs:\n" + z}, {"role": "assistant", "content": "Verstanden."}] + nachrichten[schnitt:]

def gecachte_tokens(u):
    details = getattr(u, "prompt_tokens_details", None)
    return (getattr(details, "cached_tokens", 0) or 0) if details else 0

class Gespraech:
    def __init__(self, system=None, tools=None, schwelle=None, max_tool_zeichen=None):
        self.system, self.tools, self.schwelle = system or SYSTEM, tools or TOOLS, schwelle
        self.max_tool_zeichen = max_tool_zeichen          # kürzt Tool-Ergebnisse, bevor sie in den Verlauf kommen
        self.nachrichten, self.protokoll = [], []

    def frage(self, text, max_runden=8):
        if self.schwelle and self.protokoll and self.protokoll[-1]["kontext"] > self.schwelle:
            self.nachrichten = kompaktiere(self.nachrichten)
            print("   🗜️  kompaktiert")
        self.nachrichten.append({"role": "user", "content": text})
        z = {"input": 0, "cache_write": 0, "cache_read": 0, "output": 0}
        for _ in range(max_runden):
            a = client.chat.completions.create(model=MODEL, max_tokens=1000, tools=als_tools(self.tools),
                                               messages=[{"role": "system", "content": self.system}] + self.nachrichten)
            u = a.usage
            z["cache_read"] += gecachte_tokens(u); z["input"] += u.prompt_tokens - gecachte_tokens(u); z["output"] += u.completion_tokens
            kontext = u.prompt_tokens
            msg = a.choices[0].message
            self.nachrichten.append(assistent_nachricht(msg))
            if not msg.tool_calls:
                break
            for tc in msg.tool_calls:
                r, fehler = fuehre_tool_aus(tc.function.name, argumente(tc))
                n = tool_nachricht(tc.id, r, fehler)
                if self.max_tool_zeichen and len(n["content"]) > self.max_tool_zeichen:
                    n["content"] = n["content"][:self.max_tool_zeichen] + " …[gekürzt]"
                self.nachrichten.append(n)
        self.protokoll.append({"frage": text, **z, "kontext": kontext})
        return msg.content or ""

    def kosten(self):
        return sum(kosten_cent(p) for p in self.protokoll)
print("Gesprächs-Agent bereit")

## 3 · Ein typisches Gespräch

Passt die acht Fragen an euren Track und eure User Stories an. Die letzte Frage prüft, ob nach einer Kompaktierung noch alles Wichtige bekannt ist.

In [ ]:
GESPRAECH = {
    "A": ["Hallo! Welche Workshops stehen an?", "Was steht in der Satzung zu Engagementbescheinigungen?",
          "Hat Sara Lindner (D-007) Anspruch auf eine Bescheinigung?", "Wie hoch war das Budget für das Sommerfest, und was wurde ausgegeben?",
          "Was wurde zum Hackathon beschlossen?", "Wie lange vorher muss zur Mitgliederversammlung eingeladen werden?",
          "Welche Termine gibt es im Januar?", "Fass bitte zusammen, was wir heute besprochen haben und was noch zu tun ist."],
    "B": ["Guten Morgen. Welche Tickets sind offen?", "Wie ist der Status von M-03?", "Was sagt das Handbuch zu Vibration an der Drehmaschine?",
          "Welches Fett brauche ich für die Spindellager?", "Wie ist der Status von M-06?", "Was bedeutet Fehler E21?",
          "Wo liegt das Ersatz-Spannfutter?", "Fass bitte zusammen, was wir heute besprochen haben und was noch zu tun ist."],
}[TRACK]
# ▶ GESPRAECH = [...]   eigene Fragen

def fuehre_gespraech(g, fragen, zeigen=False):
    for f in fragen:
        antwort = g.frage(f)
        if zeigen:
            print(f"❓ {f}\n💬 {antwort[:250]}\n")
    return antwort

## 4 · Vorher und nachher messen

In [ ]:
varianten = {
    "vorher": Gespraech(),
    "gekürzte Tool-Ergebnisse": Gespraech(max_tool_zeichen=1500),
    "gekürzt und kompaktiert": Gespraech(max_tool_zeichen=1500, schwelle=8000),
}
letzte_antworten = {}
for name, g in varianten.items():
    print(f"=== {name} ===")
    letzte_antworten[name] = fuehre_gespraech(g, GESPRAECH, zeigen=(name == "vorher"))

fig, ax = plt.subplots(figsize=(8, 3.5))
for name, g in varianten.items():
    ax.plot(range(1, len(g.protokoll) + 1), [p["kontext"] for p in g.protokoll], marker="o", label=f"{name}: {sum(p['input'] + p['cache_read'] + p['output'] for p in g.protokoll):,} Tokens, kommerziell {g.kosten():.2f} Cent".replace(",", "."))
ax.set_xlabel("Frage"); ax.set_ylabel("Kontext (Tokens)"); ax.legend(); ax.grid(alpha=0.3); plt.show()
for name in varianten:
    print(f"\n{name}: letzte Antwort\n💬 {letzte_antworten[name][:400]}")

**Prüft von Hand:** Ist die Zusammenfassung am Ende bei allen Varianten vollständig? Notiert Tokens pro Gespräch und eure Einschätzung der Qualität.

## 5 · Memory mit Datenschutzregel

In [ ]:
MEMORY_DATEI = "memory.json"
if TRACK == "A":
    PERSONEN = [m["vorname"] for m in lies_csv("mitglieder.csv") if m["nachname"]] + [m["nachname"] for m in lies_csv("mitglieder.csv") if m["nachname"]]
else:
    PERSONEN = sorted({z["schichtleitung"].split()[-1] for z in lies_csv("schichtbuch.csv")})
SENSIBEL = ["krank", "arzt", "privat", "leistungsbewertung", "beurteilung", "abmahnung", "fehlzeit", "telefon", "e-mail", "adresse"]

def lade_memory():
    return json.load(open(MEMORY_DATEI, encoding="utf-8")) if os.path.exists(MEMORY_DATEI) else []

def remember(thema, inhalt):
    text = thema + " " + inhalt
    if any(re.search(rf"\b{re.escape(p)}\b", text, re.IGNORECASE) for p in PERSONEN) or \
       any(re.search(rf"\b{re.escape(w)}", text, re.IGNORECASE) for w in SENSIBEL):
        raise ValueError("Nicht gespeichert: Die Notiz enthält personenbezogene oder sensible Angaben.")
    e = lade_memory()
    neu = {"id": max((x["id"] for x in e), default=0) + 1, "zeit": datetime.datetime.now().isoformat(timespec="minutes"), "thema": thema, "inhalt": inhalt}
    json.dump(e + [neu], open(MEMORY_DATEI, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return {"gespeichert": neu["id"]}

def recall(suchbegriff):
    s = suchbegriff.lower()
    return sorted([e for e in lade_memory() if s in e["thema"].lower() or s in e["inhalt"].lower()], key=lambda e: e["id"], reverse=True)[:5]

def vergiss(id):
    e = lade_memory()
    json.dump([x for x in e if x["id"] != id], open(MEMORY_DATEI, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return {"geloescht": id}

FUNKTIONEN.update({"remember": remember, "recall": recall})
TOOLS_MEMORY = TOOLS + [
    {"name": "remember", "description": "Speichert eine Notiz dauerhaft für spätere Gespräche. Nur fachliche Befunde, Entscheidungen und offene Punkte. "
                                        "Keine personenbezogenen Daten wie Namen, Gesundheit, Kontaktdaten oder Leistungsbewertungen.",
     "parameters": {"type": "object", "properties": {"thema": {"type": "string"}, "inhalt": {"type": "string"}}, "required": ["thema", "inhalt"]}},
    {"name": "recall", "description": "Sucht in gespeicherten Notizen aus früheren Gesprächen.",
     "parameters": {"type": "object", "properties": {"suchbegriff": {"type": "string"}}, "required": ["suchbegriff"]}},
]
SYSTEM_MEMORY = SYSTEM + " Du hast ein Langzeitgedächtnis (remember, recall). Prüfe bei Bedarf mit recall, ob es frühere Notizen gibt."
print(len(PERSONEN), "Namen im Datenschutzfilter")

In [ ]:
MERK_FRAGEN = {
    "A": ["Merke dir: Der Hackathon braucht noch die Zustimmung der Mitgliederversammlung zum Budget.",
          "Merke dir auch, dass Jonas Weber nächste Woche krank ist.",
          "Gibt es offene Punkte aus früheren Gesprächen zum Hackathon?"],
    "B": ["Merke dir: An M-03 ist der Lagerwechsel fürs Wochenende geplant, Ersatzteil HTD-1100 prüfen.",
          "Merke dir auch, dass Yilmaz heute krank war.",
          "Gibt es Notizen aus früheren Gesprächen zu M-03?"],
}[TRACK]
if os.path.exists(MEMORY_DATEI): os.remove(MEMORY_DATEI)
tag1 = Gespraech(system=SYSTEM_MEMORY, tools=TOOLS_MEMORY)
for f in MERK_FRAGEN[:2]:
    print(f"❓ {f}\n💬 {tag1.frage(f)}\n")
tag2 = Gespraech(system=SYSTEM_MEMORY, tools=TOOLS_MEMORY)
print(f"— neues Gespräch —\n❓ {MERK_FRAGEN[2]}\n💬 {tag2.frage(MERK_FRAGEN[2])}")
print("\nInhalt von memory.json:", lade_memory())

## 6 · Ergebnisse speichern

In [ ]:
os.makedirs("evals", exist_ok=True)
zeit = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
bericht = {name: {"kosten_cent": g.kosten(), "kontext_letzte_frage": g.protokoll[-1]["kontext"],
                  "tokens": {a: sum(p[a] for p in g.protokoll) for a in PREISE}} for name, g in varianten.items()}
json.dump({"zeit": zeit, "notiz": "Woche 11: Kosten pro Gespräch", "varianten": bericht, "memory": lade_memory()},
          open(f"evals/kontext_{zeit}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for name, b in bericht.items():
    print(f"{name:32s} kommerziell {b['kosten_cent']:.2f} Cent pro Gespräch (Referenz)")

*Auswertung (für das Repo):* Kosten pro Gespräch vorher und nachher, Qualität der Schlusszusammenfassung, was das Memory gespeichert und abgelehnt hat, und eure Entscheidung, welche Maßnahmen ihr behaltet.

- 